# 04. Data Leakage Audit & Financial Portfolio Allocation
> **Quantifying post-release leakage and simulating net portfolio ROI**

Part of the **GreenlightML** open-source film risk modeling and portfolio optimization engine.


> **Draft / not validated:** This notebook is a scaffold. The current feature builder must be replaced with train-fitted preprocessing before any score is treated as out-of-sample evidence.

> **Draft / not validated:** This notebook is a future-stage scaffold. Raw rows are split before any learned preprocessing (fit on train only), but the target/population/decision contract is not frozen, so scores here are not accepted evidence.

## 1. Data Setup & Controlled Leakage Experiment
Evaluate the artificial performance leap when post-release features contaminate models.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc

sys.path.append(os.path.abspath('..'))
from src.data import load_and_merge_data, filter_reported_financials
from src.features import FeatureEncoder
from src.models import get_logistic_regression, get_random_forest
from src.metrics import evaluate_model

df_reported = filter_reported_financials(load_and_merge_data())

# Split the raw rows FIRST; leakage columns are attached only after the split.
train_df, test_df = train_test_split(
    df_reported, test_size=0.2, random_state=42, stratify=df_reported['target_nominal']
)
encoder = FeatureEncoder(max_tfidf_features=100).fit(train_df)
X_tr_pre = encoder.transform(train_df)
X_te_pre = encoder.transform(test_df)
X_tr_leak = encoder.transform(train_df, include_leakage=True)
X_te_leak = encoder.transform(test_df, include_leakage=True)
y_tr, y_te = train_df['target_nominal'], test_df['target_nominal']
test_slice = test_df

## 2. Measuring the Artificial Data Leakage Lift
Train identical models on pre-release vs. leaked feature sets.


In [ ]:
# Pre-release model
sc_pre = StandardScaler()
lr_pre = get_logistic_regression()
lr_pre.fit(sc_pre.fit_transform(X_tr_pre), y_tr)
prob_pre = lr_pre.predict_proba(sc_pre.transform(X_te_pre))[:, 1]

# Contaminated (Leaked) model
sc_leak = StandardScaler()
lr_leak = get_logistic_regression()
lr_leak.fit(sc_leak.fit_transform(X_tr_leak), y_tr)
prob_leak = lr_leak.predict_proba(sc_leak.transform(X_te_leak))[:, 1]

fpr_pre, tpr_pre, _ = roc_curve(y_te, prob_pre)
fpr_leak, tpr_leak, _ = roc_curve(y_te, prob_leak)

plt.figure(figsize=(7, 5))
plt.plot(fpr_pre, tpr_pre, label=f'Clean Pre-Release (AUC = {auc(fpr_pre, tpr_pre):.3f})', color='#10b981', lw=2)
plt.plot(fpr_leak, tpr_leak, label=f'Contaminated Leakage (AUC = {auc(fpr_leak, tpr_leak):.3f})', color='#ef4444', lw=2, linestyle='--')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
plt.title("ROC Curves: Clean Pre-Release vs. Post-Release Data Leakage")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()


## 3. Financial proxy analysis (not net portfolio ROI)

Compare a model-driven allocation against the passive market index. The financial figure is a gross-to-budget surplus proxy, not studio net ROI; see `docs/METRICS_AND_ROI.md`.

In [ ]:
# Passive Market Index (Fund all 646 holdout titles)
y_all = np.ones_like(y_te)
res_market = evaluate_model("Passive Market Index", "Passive", y_te, y_all, None, test_slice)

# Model-driven greenlighting (Logistic Regression)
res_lr_clean = evaluate_model("Logistic Regression", "Pre-release", y_te, lr_pre.predict(sc_pre.transform(X_te_pre)), prob_pre, test_slice)
res_lr_leak = evaluate_model("Logistic Regression (LEAKED)", "Post-release", y_te, lr_leak.predict(sc_leak.transform(X_te_leak)), prob_leak, test_slice)

pd.DataFrame([res_market, res_lr_clean, res_lr_leak])
